# Bump test recent history

In [ ]:
# Times Square parameters
day_obs = "20260701"  # day_obs in format yyyymmdd
days_to_plot = 5

In [ ]:
import asyncio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colors as mcolors
from matplotlib import dates as mdates
from astropy.time import Time, TimeDelta
from lsst.ts.xml.tables.m1m3 import FATable, FAIndex, force_actuator_from_id, actuator_id_to_index
from lsst_efd_client import EfdClient
from lsst.ts.xml.enums.MTM1M3 import BumpTest

In [ ]:
%matplotlib inline
client = EfdClient("usdf_efd")
start = Time.strptime(day_obs, "%Y%m%d", scale="utc")
end = start + TimeDelta(days_to_plot, format="jd")

In [ ]:
def max_error(errors):
    return np.max(np.abs(errors))


def rms_error(times, errors):
    in_window = (((times > 3.0) & (times < 4.0))
                 | ((times > 10.0) & (times < 11.0)))
    if not np.any(in_window):
        return np.nan
    return np.sqrt(np.mean(errors[in_window]**2))

In [ ]:
async def calc_bumps_and_errors(client, bump, bt_result, follow):
    BUMP_TEST_DURATION = 14.0  # seconds
    measured_forces_times = []
    following_error_values = []
    t_starts = []
    results = bt_result[bt_result[bump] == BumpTest.TESTINGPOSITIVE]
    for timestamp in results.index:
        t_start = Time(
            (timestamp - pd.Timedelta(seconds=1.0)).timestamp(),
            format="unix_tai",
            scale="tai",
        )
        t_starts.append(t_start)
        t_end = Time(
            t_start + TimeDelta(BUMP_TEST_DURATION, format="sec"),
            format="unix_tai",
            scale="tai",
        )

        measured_forces = await client.select_time_series(\
                    "lsst.sal.MTM1M3.forceActuatorData", \
                    [follow, "timestamp"], t_start.utc, t_end.utc)
        
        t0 = measured_forces["timestamp"].values[0]
        measured_forces["timestamp"] -= t0

        # It is easier/faster to work with arrays
        measured_forces_time = measured_forces["timestamp"].values
        measured_forces_times.append(measured_forces_time)
        following_error_value = measured_forces[follow].values
        following_error_values.append(following_error_value)

    times = []
    max_errors = []
    rms_errors = []
    for i in range(len(measured_forces_times)):
        times.append(t_starts[i])
        max_errors.append(max_error(following_error_values[i]))
        rms_errors.append(rms_error(measured_forces_times[i], following_error_values[i]))
    return [times, rms_errors, max_errors]

In [ ]:
async def actuator_error(client, fa_id, bt_results):
    # Grab the Force Actuator Data from its ID
    fa_data = force_actuator_from_id(fa_id)
    
    # First the primary forces
    bump = f"primaryTest{fa_data.index}"
    follow = f"primaryCylinderFollowingError{fa_data.index}"
    [ptimes, prms_errors, pmax_errors] = \
        await calc_bumps_and_errors(client, bump, bt_results, follow)

    # Now the secondary forces
    if fa_data.actuator_type.name == "DAA":
        bump = f"secondaryTest{fa_data.s_index}"
        follow = f"secondaryCylinderFollowingError{fa_data.s_index}"
        [stimes, srms_errors, smax_errors] = \
            await calc_bumps_and_errors(client, bump, bt_results, follow)
    else:
        stimes = []; srms_errors = []; smax_errors = []

    return [ptimes, prms_errors, pmax_errors, stimes, srms_errors, smax_errors]

In [ ]:
async def plot_failed_actuators(client, ids, bumps, start, end):
    cmap = mcolors.ListedColormap(["cornflowerblue", "gold", "crimson"])
    fig, axs = plt.subplots(2, 2, figsize=(12, 10))
    plt.suptitle("Actuator Recent History\nNot plotted -> All within acceptable limit", fontsize=18)
    plot_list = [[0, 0, "Primary RMS Errors", 0, 1, 5, 10],
                 [1, 0, "Primary Max Errors", 0, 2, 100, 200],
                 [0, 1, "Secondary RMS Errors", 3, 4, 7.0, 14.0],
                 [1, 1, "Secondary Max Errors", 3, 5, 140, 280]]
    ns = np.zeros(4)  # Number of bad ids per plot
    bad_ids = [[], [], [], []]  # List of bad_ids per plot
    s = [[], [], [], []]  # List of scatter plots
    
    for actuator_id in ids:
        data = await actuator_error(client, actuator_id, bumps)
        for m, [xplot, yplot, title, time_index, error_index, yellow_limit, red_limit] in enumerate(plot_list):
            errors = np.array(data[error_index])
            times = data[time_index]
            tai_times = np.array([Time(time, format="isot").datetime for time in times])
            
            if len(errors) > 0:
                max_error = np.max(errors)
            else:
                max_error = 0.0
            if max_error > yellow_limit:
                bad_ids[m].append(actuator_id)
                yaxis = np.array([ns[m]] * len(times))
            else:
                continue

            boundaries = [0, yellow_limit, red_limit, red_limit * 2]
            norm = mcolors.BoundaryNorm(boundaries, cmap.N)

            # Sorting by error value to plot red on top of yellow on top of green points
            idx = np.argsort(errors)
            s[m] = axs[xplot][yplot].scatter(tai_times[idx], yaxis[idx], c=errors[idx], cmap=cmap, norm=norm)
            ns[m] += 1

    for m, [xplot, yplot, title, time_index, error_index, yellow_limit, red_limit] in enumerate(plot_list):
        axs[xplot][yplot].set_title(title)

        # Setup major xticks per day and minor xticks every 4 hours 
        axs[xplot][yplot].set_xlim(start.datetime, end.datetime)
        axs[xplot][yplot].xaxis.set_major_locator(mdates.DayLocator(tz=None))
        axs[xplot][yplot].xaxis.set_minor_locator(mdates.HourLocator(byhour=range(0, 24, 4)))
        axs[xplot][yplot].xaxis.set_major_formatter(mdates.DateFormatter('%Y\n%m-%d'))
        axs[xplot][yplot].tick_params(axis='x', which='major', length=6, labelsize=9)
        axs[xplot][yplot].tick_params(axis='x', which='minor', length=3, color='gray')
        axs[xplot][yplot].grid(visible=True, which='major', axis='x', color='black', alpha=0.2, linestyle='-')
        axs[xplot][yplot].grid(visible=True, which='minor', axis='x', color='black', alpha=0.2, linestyle='--')
        axs[xplot][yplot].set_axisbelow(True)
        
        axs[xplot][yplot].set_yticks(range(len(bad_ids[m])), bad_ids[m])
        axs[xplot][yplot].set_ylabel("Actuator ID")
        if s[m]:
            cbar = fig.colorbar(s[m], ax=axs[xplot][yplot],
                                orientation="vertical", shrink=0.75, extend="max")
            cbar.set_label("Error [N]", fontsize=10)

    plt.show()

In [ ]:
bumps = await client.select_time_series(\
                    "lsst.sal.MTM1M3.logevent_forceActuatorBumpTestStatus", \
                    ["*"], start, end)
ids = [actuator.actuator_id for actuator in FATable]
if not bumps.empty:
    await plot_failed_actuators(client, ids, bumps, start, end)
else:
    print("Query returned no data. Nothing to plot.")